 # Analysis Overview Flow PBMC Frequency Analysis
This notebook contains the following analyses:

## 1. NDMM subjects on VRd vs Healthy Donors

- Comparative analyses between participants in **ARM1** of this study and **healthy donor** controls.
- Evaluation of differences across relevant clinical and/or molecular features.
- Statistical testing and visualization to assess group-level variation.

## 2. Paired Analyses Across Clinical Time Points (VRd)

- Longitudinal, paired analyses within **VRd** participants.
- Comparison of matched samples collected at different clinical time points.
- Within-subject statistical testing to evaluate temporal changes over the course of the study.
- Visualization of trajectories and paired differences.


In [1]:
suppressPackageStartupMessages({
  library(data.table)
  library(dplyr)
  library(purrr)
  library(ggplot2)
  library(ggpubr)
  library(rstatix)
  library(forcats)
  library(tidyr)
  library(ggrepel)
  library(rlang)
  library(scales)
  
})
options(repr.plot.width = 11, repr.plot.height = 6)

plTheme = theme_classic() +
    theme(
     strip.text = element_text(size = 12, hjust = 0, face = "bold"),
      strip.background = element_blank(),
      axis.title = element_text(size = 12, face = "bold"),
      axis.text = element_text(size = 10),
      legend.position = "bottom",
      legend.text = element_text(size = 11),
      panel.spacing = grid::unit(0.5, "cm")
    )

## Load Metadata

In [2]:
### Load Metadata 
meta = readxl::read_xlsx('../data/flow/ndmm-rrmm-metadata-2025.xlsx',sheet = 1)
meta$subject.subjectGuid <- meta$Subject

### delete first 3 rows as 
### these are reference ranges 
### and descriptors for each column 
meta = meta[-c(1:3),]

### Refactor all 'BRI' subjects as healthy
meta[meta$Cohort %in% c('BR1','BR2'),]$Cohort <- 'Healthy'
table(meta$Cohort)


    FH1 Healthy 
    197      34 

## Load Compositional Data

### Combine all files and summarize experiments

In [3]:
### Load combined files 
mmFiles = fread('../data/flow/output/aggregated_flowdata.csv')

### Join metadata with data files
mmFiles <- dplyr::left_join(mmFiles, distinct(meta[,c('subject.subjectGuid','Sub_Cohort')]),
                      by='subject.subjectGuid')

mmFiles$label.visitDetails <- factor(mmFiles$label.visitDetails,
                                    levels=c('PreTx','PI2C','EI','ASCT60d','ASCT1y','ASCT2y','Healthy'))

#write.csv(mmFiles, '../data/flow/output/aggregated_flow.csv')

# Analysis
## VRD NDMM Patients vs. Healthy

In [4]:
## Get cell pop names
uniqueCells = unique(mmFiles$aifi_label_l2)

## Include healthy subjects
mm_patients = mmFiles[Sub_Cohort %in% c('Healthy','VRd')]

test_timePoint <- function(time='PreTx'){
    ## repeat for each tim
    preTX = mm_patients[label.visitDetails %in% c(time,'Healthy')]
    res <- lapply(uniqueCells,
                  function(x){
                      tmp=preTX[aifi_label_l2==x]
                      res = data.frame(
                          VRD = median(tmp[Sub_Cohort=='VRd']$pseudo_alc, na.rm=T),
                          Healthy =  median(tmp[Sub_Cohort=='Healthy']$pseudo_alc, na.rm=T),
                          panel= unique(tmp$panel),
                          Time=time,
                          Cell=x,
                          Pval=wilcox.test(tmp[Sub_Cohort=='VRd']$pseudo_alc,
                                           tmp[Sub_Cohort=='Healthy']$pseudo_alc)$p.value)
                      }
                  )
    res = rbindlist(res)
    res$FDR = p.adjust(res$Pval, method='fdr')
    res$Log2FC = log2(res$VRD) - log2(res$Healthy)
    return(res)    
}

healthy_comparisons <- rbindlist(lapply(c('PreTx','PI2C','EI','ASCT60d','ASCT1y','ASCT2y'),
       function(x) test_timePoint(x)
       ))

Warning message in wilcox.test.default(tmp[Sub_Cohort == "VRd"]$pseudo_alc, tmp[Sub_Cohort == :
“cannot compute exact p-value with ties”
Warning message in wilcox.test.default(tmp[Sub_Cohort == "VRd"]$pseudo_alc, tmp[Sub_Cohort == :
“cannot compute exact p-value with ties”
Warning message in wilcox.test.default(tmp[Sub_Cohort == "VRd"]$pseudo_alc, tmp[Sub_Cohort == :
“cannot compute exact p-value with ties”
Warning message in wilcox.test.default(tmp[Sub_Cohort == "VRd"]$pseudo_alc, tmp[Sub_Cohort == :
“cannot compute exact p-value with ties”
Warning message in wilcox.test.default(tmp[Sub_Cohort == "VRd"]$pseudo_alc, tmp[Sub_Cohort == :
“cannot compute exact p-value with ties”


## VRD Subjects paired analyses

In [5]:
## Get cell pop names
mm_patients = mmFiles[Sub_Cohort =='VRd']
timepoints = data.frame(
    t1=c('PreTx','PI2C','PreTx', 'EI','ASCT60d','ASCT1y', 'EI','EI'),
    t2=c('PI2C','EI','EI', 'ASCT60d','ASCT1y','ASCT2y','ASCT60d','ASCT1y'))

testPre_Post <- function(t1, t2){
    timePointDF = mm_patients[label.visitDetails %in% c(t1,t2), 
                              c('label.visitDetails','pseudo_alc','subject.subjectGuid','aifi_label_l2')]


    resList = lapply(uniqueCells,
           function(x){
            tmpDf = timePointDF[aifi_label_l2==x]
            tmpDf = dcast(tmpDf, subject.subjectGuid ~ label.visitDetails, value.var='pseudo_alc')
            tmpDf = tmpDf[complete.cases(tmpDf)]
            tmpDf = tmpDf[, c('subject.subjectGuid',t1,t2), with=F]
            tmpDf
            g1 = tmpDf[,t1, with=F][[1]]
            g2 =tmpDf[,t2, with=F][[1]]
            
            res = data.frame(
              T1_median = median(g1),
              T2_median =  median(g2),
              T1=t1,
              T2=t2,
              Cell=x,
              Pval=wilcox.test(g1,g2, paired=T)$p.value
                )
            res}
           )
    resList = rbindlist(resList)
    resList$FDR = p.adjust(resList$Pval, method='fdr')
    resList$Log2FC = log2(resList$T1_median) - log2(resList$T2_median)
    resList$Contrast = paste(t1,t2,sep='-')
    resList
    
}

pairedComparisons <- rbindlist(lapply(1:nrow(timepoints),
       function(x)
        testPre_Post(timepoints$t1[x],
                     timepoints$t2[x])
       ))
                               

Warning message in wilcox.test.default(g1, g2, paired = T):
“cannot compute exact p-value with zeroes”
Warning message in wilcox.test.default(g1, g2, paired = T):
“cannot compute exact p-value with zeroes”


In [6]:
write.csv(healthy_comparisons, file='../data/flow/output/vrd_healthy_comparisons.csv')
write.csv(pairedComparisons, file='../data/flow/output/vrd_longitudinal_comparisons.csv')